# Fase 3 · M00: Índice Feature Engineering

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M00 — Índice |

---

## 🎯 Qué hace

Genera la página índice HTML de Fase 3 con la lista de módulos y su descripción.

## 📋 Requisitos

- `src/html/render.py` con `render_pagina_desde_fichero()`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase3/fase3_index.html` | Índice navegable de Fase 3 |

## 🔄 Flujo

```
render_pagina_desde_fichero() → HTML → escribe en disco
```

## ➡️ Siguiente

`f3_m01_validacion.ipynb` — validación y limpieza del dataset procesado


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import (
    RUTA_HTML, RUTA_PROCESSED, RUTA_FEATURES, RUTA_AUTOML,
    info_entorno
)
from src.utils import crear_directorios
from src.html import (
    generar_kpis_html,
    generar_tarjetas_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

RUTA_FASE3 = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FASE3])

info_entorno()

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: LEER DATOS REALES DEL PROYECTO (dinámico, sin hardcodes)
# ============================================================================
# Lee df_expediente_features (intermedio, M03) y dataset_final_tfm (D_strict, M05)
# Si no existen aún (primera ejecución), muestra aviso y usa None.
# Los KPIs se adaptan automáticamente.
# ============================================================================

# Dataset intermedio (M03) — para mostrar volumen de expedientes
ruta_features = RUTA_FEATURES / 'df_expediente_features.parquet'
if ruta_features.exists():
    df_tmp = pd.read_parquet(ruta_features, columns=['cred_superados_anio_1er'])
    N_EXPEDIENTES = len(df_tmp)
    del df_tmp
    print(f'✓ Expedientes: {N_EXPEDIENTES:,}')
else:
    N_EXPEDIENTES = None
    print('⚠️  df_expediente_features no encontrado — ejecutar M01→M03 primero')

# Dataset D_strict (M05) — para mostrar features finales
ruta_dstrict = RUTA_AUTOML / 'dataset_final_tfm.parquet'
if ruta_dstrict.exists():
    df_ds = pd.read_parquet(ruta_dstrict)
    N_FEATURES = df_ds.shape[1] - 1  # sin target
    TASA_ABANDONO = f'{(df_ds["abandono"]==1).mean()*100:.1f}%'
    del df_ds
    print(f'✓ Features D_strict: {N_FEATURES}')
    print(f'✓ Tasa abandono: {TASA_ABANDONO}')
else:
    N_FEATURES = None
    TASA_ABANDONO = None
    print('⚠️  dataset_final_tfm no encontrado — ejecutar M01→M05 primero')

# Formatear para KPIs
kpi_expedientes = f'{N_EXPEDIENTES:,}' if N_EXPEDIENTES else '—'
kpi_features = str(N_FEATURES) if N_FEATURES else '—'
kpi_abandono = TASA_ABANDONO if TASA_ABANDONO else '—'


✓ Expedientes: 33,621


✓ Features D_strict: 19
✓ Tasa abandono: 29.2%


In [3]:
# ============================================================================
# CELDA 3: DATOS DE LA PÁGINA (dinámicos)
# ============================================================================

KPIS = [
    {'valor': kpi_expedientes,  'titulo': 'Expedientes'},
    {'valor': '9',              'titulo': 'Módulos'},
    {'valor': kpi_features,     'titulo': 'Features D_strict'},
    {'valor': kpi_abandono,     'titulo': 'Tasa abandono'},
]

MODULOS = [
    {
        'archivo': 'm01_validacion.html',
        'emoji': '✅',
        'titulo': 'M01: Validación y Limpieza',
        'desc': 'Correcciones P1-P8, orden_preferencia, anios_gap, dataset longitudinal.',
        'color': '#3182ce'
    },
    {
        'archivo': 'm02_agregacion.html',
        'emoji': '🔗',
        'titulo': 'M02: Agregación por Expediente',
        'desc': '1 fila por alumno. Calcula cred_repetidos, n_anios_trabajando y más.',
        'color': '#38a169'
    },
    {
        'archivo': 'm03_features.html',
        'emoji': '🧪',
        'titulo': 'M03: Features Derivadas',
        'desc': 'duracion_real, mejora_notas, estabilidad_academica, anios_sin_beca.',
        'color': '#805ad5'
    },
    {
        'archivo': 'm04_encoding.html',
        'emoji': '🏷️',
        'titulo': 'M04: Encoding Categóricas',
        'desc': 'M04a: 100% numérico sin hardcodes. M04b: Mixto para CatBoost.',
        'color': '#ed8936'
    },
    {
        'archivo': 'm05_target_export.html',
        'emoji': '🎯',
        'titulo': 'M05: Target y Export',
        'desc': 'Target abandono, target encoding de titulacion, D y D_strict.',
        'color': '#e53e3e'
    },
    {
        'archivo': 'm06_alerta_temprana.html',
        'emoji': '⚠️',
        'titulo': 'M06: Baselines de Validación',
        'desc': 'Modelos baseline sobre D y D_strict. Feature importance con XGBoost.',
        'color': '#dd6b20'
    },
    {
        'archivo': 'm07_validacion.html',
        'emoji': '🔍',
        'titulo': 'M07: Validación Completa',
        'desc': 'Estabilidad con 5 seeds, comparativa baselines, leakage residual.',
        'color': '#2b6cb0'
    },
    {
        'archivo': 'm08_auditoria.html',
        'emoji': '📋',
        'titulo': 'M08: Auditoría y Cierre',
        'desc': 'Auditoría feature a feature, diccionario, df_eda_final para Fase 4.',
        'color': '#276749'
    },
    {
        'archivo': 'm09_perfiles_riesgo.html',
        'emoji': '👤',
        'titulo': 'M09: Perfiles de Riesgo',
        'desc': 'Score de riesgo ponderado. Segmentación alto/medio/bajo por rama, beca y acceso.',
        'color': '#6b46c1'
    },
]

print(f'✅ {len(KPIS)} KPIs, {len(MODULOS)} módulos')


✅ 4 KPIs, 9 módulos


In [4]:
# ============================================================================
# CELDA 4: GENERAR HTML
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase3',
    modulo_activo='indice'
)

s_resumen = generar_seccion_html(
    titulo='Resumen de la Fase',
    contenido=f'''
        <p>Transformación del dataset de <strong>{kpi_expedientes} expedientes</strong>
        a formato listo para modelado ML.</p>
        <p>8 módulos: validación (P1-P8), agregación, features derivadas,
        encoding sin hardcodes, target encoding de titulación, baselines y auditoría.</p>
        <div style="background:#EBF8FF;padding:15px;border-radius:8px;margin-top:15px;border-left:4px solid #3182ce;">
            <strong>Dataset de producción:</strong> D_strict — {kpi_features} features + target abandono ({kpi_abandono})
        </div>
        <div style="background:#F0FFF4;padding:15px;border-radius:8px;margin-top:10px;border-left:4px solid #38a169;">
            <strong>Salidas:</strong><br>
            <code>data/automl/dataset_final_tfm.parquet</code> — D_strict para modelado<br>
            <code>data/automl/df_exp_automl_D.parquet</code> — D completo<br>
            <code>data/04_eda/df_eda_final.parquet</code> — EDA con texto legible
        </div>
    ''',
    icono='⚙️'
)

s_modulos = generar_seccion_html(
    titulo='Módulos',
    contenido=generar_tarjetas_html([
        {'titulo': m['titulo'], 'descripcion': m['desc'],
         'emoji': m['emoji'], 'link': m['archivo'],
         'link_texto': 'Abrir módulo →', 'color': m['color']}
        for m in MODULOS
    ]),
    icono='📦'
)

contenido_html = generar_kpis_html(KPIS) + s_resumen + s_modulos

html_completo = render_pagina_desde_fichero(
    'f3_m00_indice.ipynb',
    contenido_html,
    carpeta_notebook='fase3_features'
)

ruta_html = RUTA_FASE3 / 'fase3_index.html'
guardar_html(html_completo, ruta_html)
print(f'\n✅ HTML generado: {ruta_html}')


GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\fase3_index.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\fase3_index.html


In [5]:
# ============================================================================
# CELDA 5: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F3-M00 COMPLETADO')
print('=' * 60)
print(f'\n📋 HTML: {ruta_html}')
print(f'📊 {len(KPIS)} KPIs | {len(MODULOS)} módulos')
print(f'📌 Expedientes: {kpi_expedientes}')
print(f'🎯 Features D_strict: {kpi_features}')
print(f'📉 Tasa abandono: {kpi_abandono}')
print(f'\n🎯 Siguiente: f3_m01_validacion.ipynb')
print('=' * 60)



✅ F3-M00 COMPLETADO

📋 HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\fase3_index.html
📊 4 KPIs | 9 módulos
📌 Expedientes: 33,621
🎯 Features D_strict: 19
📉 Tasa abandono: 29.2%

🎯 Siguiente: f3_m01_validacion.ipynb
